# 09 SmolVLA 从 smoke 到正式训练

            SmolVLA 同时读取图像、语言和机器人状态。这里使用与 ACT 相同的数据根目录，但必须检查红杯、蓝杯指令是否都存在，不能只看总体成功率。

            这一节从 LeRobot 数据检查开始，依次完成 2-step smoke、正式训练、ROCm 资源观察和 checkpoint 定位。所有长任务默认关闭，先确认生成的配置，再显式打开运行开关。


In [1]:

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    override = (
        os.environ.get("AMD_TOPIC_ROOT")
        or os.environ.get("NOTEBOOK_TOPIC_ROOT")
        or os.environ.get("TOPIC_ROOT")
    )
    roots = [Path(override).expanduser()] if override else []
    cwd = Path.cwd().resolve()
    roots.extend([cwd, *cwd.parents])

    candidates = []
    for root in roots:
        candidates.extend(
            [
                root,
                root / "16-专题组队学习" / "04-AMD-ROCm策略复刻专题",
                root / "04-AMD-ROCm策略复刻专题",
            ]
        )
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError(
        "找不到 AMD ROCm 专题目录。请从仓库根目录、专题目录启动 Jupyter，"
        "或设置 AMD_TOPIC_ROOT。"
    )


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", TOPIC_ROOT / "external" / "mujoco_pnp")
).expanduser()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", TOPIC_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


TOPIC_ROOT = $NOTEBOOK_TOPIC_ROOT
PROJECT_ROOT = $PROJECT_ROOT
DATA_ROOT = $PROJECT_ROOT
OUTPUT_ROOT = $OUTPUT_ROOT
MODEL_ROOT = $MODEL_ROOT


In [2]:

try:
    from IPython.display import Image, Markdown, Video, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"

    def Video(filename=None, embed=False, width=None, **kwargs):
        return f"[video] {filename}"


def show_video(filename, title=None, width=960):
    path = ASSET_DIR / filename
    if title:
        display(Markdown(f"**{title}**"))
    if not path.exists():
        print(f"缺少视频素材：{path}")
        return
    try:
        display(Video(filename=str(path), embed=True, width=width, html_attributes="controls muted"))
    except TypeError:
        display(Video(filename=str(path), embed=True, width=width))


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        if path.suffix.lower() in {".mp4", ".webm", ".mov", ".m4v"}:
            show_video(filename, width=width)
            return
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


In [3]:
import shlex

try:
    import yaml
except ImportError as exc:
    raise RuntimeError("当前环境缺少 PyYAML，请先执行 pip install pyyaml。") from exc


def require_project_layout():
    required = [
        PROJECT_ROOT / "train_model.py",
        PROJECT_ROOT / "env_config.py",
        PROJECT_ROOT / "asset" / "example_scene_y2.xml",
        PROJECT_ROOT / "mujoco_env" / "y_env2.py",
    ]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "PROJECT_ROOT 不是 04mujoco 教程目录，缺少：\n"
            + "\n".join(str(path) for path in missing)
        )
    return True


def dataset_report(dataset_root):
    dataset_root = Path(dataset_root)
    info_path = dataset_root / "meta" / "info.json"
    tasks_path = dataset_root / "meta" / "tasks.jsonl"
    if not info_path.exists():
        raise FileNotFoundError(f"找不到 LeRobot 元数据：{info_path}")
    info = json.loads(info_path.read_text(encoding="utf-8"))
    tasks = []
    if tasks_path.exists():
        tasks = [
            json.loads(line)["task"]
            for line in tasks_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    features = info.get("features", {})
    rows = [
        ("repo_id", info.get("repo_id", "")),
        ("episodes", info.get("total_episodes", 0)),
        ("frames", info.get("total_frames", 0)),
        ("fps", info.get("fps", "")),
        ("state shape", features.get("observation.state", {}).get("shape")),
        ("action shape", features.get("action", {}).get("shape")),
        ("tasks", " / ".join(tasks)),
    ]
    md_table(["数据项", "读取结果"], rows)
    return info, tasks


def make_training_config(
    policy_type,
    dataset_repo_id,
    dataset_root,
    output_dir,
    steps,
    batch_size,
    chunk_size,
    n_action_steps,
    save_freq,
    seed=42,
):
    return {
        "dataset": {"repo_id": dataset_repo_id, "root": str(Path(dataset_root))},
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "save_checkpoint": True,
        "output_dir": str(Path(output_dir)),
        "batch_size": int(batch_size),
        "job_name": Path(output_dir).name,
        "resume": False,
        "seed": int(seed),
        "num_workers": 4,
        "steps": int(steps),
        "eval_freq": 0,
        "log_freq": max(1, min(50, int(steps))),
        "save_freq": int(save_freq),
        "use_policy_training_preset": True,
        "wandb": {
            "enable": False,
            "project": f"every_embodied_{policy_type}",
            "entity": None,
            "disable_artifact": True,
        },
    }


def write_training_config(name, config):
    config_dir = OUTPUT_ROOT / "configs"
    config_dir.mkdir(parents=True, exist_ok=True)
    path = config_dir / f"{name}.yaml"
    path.write_text(
        yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print("已写出配置：", path)
    return path


def training_command(config_path):
    return [
        sys.executable,
        str(PROJECT_ROOT / "train_model.py"),
        "--config_path",
        str(Path(config_path)),
    ]


def run_training(config_path, enabled=False):
    require_project_layout()
    command = training_command(config_path)
    print("$", shlex.join(command))
    if not enabled:
        print("当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。")
        return None
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{PROJECT_ROOT}:{env.get('PYTHONPATH', '')}"
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def show_rocm_resources():
    command = ["rocm-smi", "--showuse", "--showmemuse", "--showtemp"]
    print("$", shlex.join(command))
    if shutil.which(command[0]) is None:
        print("未找到 rocm-smi。确认当前机器是否安装 ROCm。")
        return
    result = subprocess.run(command, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip(), file=sys.stderr)


## Checkpoint 1：确认项目和训练数据


In [4]:
require_project_layout()
print("train_model.py =", PROJECT_ROOT / "train_model.py")
print("训练数据 =", Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language")))


train_model.py = $PROJECT_ROOT/train_model.py
训练数据 = $TRAIN_DATA_ROOT


In [5]:
candidate = Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language"))
if (candidate / "meta" / "info.json").exists():
    dataset_report(candidate)
else:
    print("训练数据还没准备好：", candidate)
    print("先完成 07_data_collection_and_audit.ipynb，或设置 TRAIN_DATA_ROOT 指向已有 LeRobot 数据集。")


| 数据项 | 读取结果 |
| --- | --- |
| repo_id |  |
| episodes | 20 |
| frames | 2621 |
| fps | 20 |
| state shape | [6] |
| action shape | [7] |
| tasks | Place the blue mug on the plate. / Place the red mug on the plate. |


## Checkpoint 2：生成 smoke 与正式训练配置


In [6]:
MODEL_TYPE = 'smolvla'
DATASET_REPO_ID = os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language")
TRAIN_DATA_ROOT = Path(
    os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language")
).expanduser()

RUN_SMOKE = False
RUN_FULL_TRAIN = False

SMOKE_OUTPUT = MODEL_ROOT / f"{MODEL_TYPE}_rocm_smoke"
FULL_OUTPUT = MODEL_ROOT / f"{MODEL_TYPE}_rocm_full"

smoke_config = make_training_config(
    policy_type=MODEL_TYPE,
    dataset_repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
    output_dir=SMOKE_OUTPUT,
    steps=2,
    batch_size=min(4, 4),
    chunk_size=5,
    n_action_steps=5,
    save_freq=2,
)
full_config = make_training_config(
    policy_type=MODEL_TYPE,
    dataset_repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
    output_dir=FULL_OUTPUT,
    steps=20000,
    batch_size=4,
    chunk_size=5,
    n_action_steps=5,
    save_freq=max(1, 20000 // 2),
)

smoke_config_path = write_training_config(f"{MODEL_TYPE}_smoke", smoke_config)
full_config_path = write_training_config(f"{MODEL_TYPE}_full", full_config)
print("smoke output =", SMOKE_OUTPUT)
print("full output =", FULL_OUTPUT)


已写出配置： $OUTPUT_ROOT/configs/smolvla_smoke.yaml
已写出配置： $OUTPUT_ROOT/configs/smolvla_full.yaml
smoke output = $MODEL_ROOT/smolvla_rocm_smoke
full output = $MODEL_ROOT/smolvla_rocm_full


配置文件会写到 `$OUTPUT_ROOT/configs`，checkpoint 写到 `$MODEL_ROOT`。这两个目录应放在容量充足的磁盘，不要写进 Git 仓库。


## Checkpoint 3：先跑 2-step smoke


In [7]:
show_rocm_resources()
run_training(smoke_config_path, enabled=RUN_SMOKE)


$ rocm-smi --showuse --showmemuse --showtemp


============================ ROCm System Management Interface ============================
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 31.0
=================================== % time GPU is busy ===================================
GPU[0]		: GPU use (%): 0
=================================== Current Memory Use ===================================
GPU[0]		: GPU Memory Allocated (VRAM%): 9
GPU[0]		: Memory Activity: N/A
GPU[0]		: Not supported on the given system
================================== End of ROCm SMI Log ===================================
$ $NOTEBOOK_PYTHON '$PROJECT_ROOT/train_model.py' --config_path '$OUTPUT_ROOT/configs/smolvla_smoke.yaml'
当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。


2-step smoke 只证明数据加载、模型构造、forward/backward、optimizer step 和 checkpoint 写出可用；它不证明模型收敛，更不能替代 MuJoCo closed-loop 成功率。


## Checkpoint 4：启动正式训练


In [8]:
run_training(full_config_path, enabled=RUN_FULL_TRAIN)


$ $NOTEBOOK_PYTHON '$PROJECT_ROOT/train_model.py' --config_path '$OUTPUT_ROOT/configs/smolvla_full.yaml'
当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。


SmolVLA 首次运行会下载基础权重。正式训练建议保存中间 checkpoint，并分别统计红杯和蓝杯的 physical_success；如果一类明显落后，先检查任务分布和 gripper 事件，再考虑 Weighted sampler。


常见恢复顺序：显存不足先减小 `batch_size`；出现 DataLoader worker / pickle 错误时把 YAML 中的 `num_workers` 改为 `0`；下载报 `401/403` 时检查账号权限和私密 `HF_TOKEN`；写 checkpoint 失败先检查 `$MODEL_ROOT` 所在磁盘。ROCm 的 MIOpen 数据库 warning 如果没有伴随训练退出可以记录后继续观察，出现 kernel 退出、NaN 或进程消失则必须停止长训练并回到 smoke。


## Checkpoint 5：定位 checkpoint 并检查资源


In [9]:
show_rocm_resources()
if (FULL_OUTPUT / "checkpoints").exists():
    checkpoints = sorted((FULL_OUTPUT / "checkpoints").glob("*/pretrained_model"))
    print("可用 checkpoint：")
    for path in checkpoints:
        print(" -", path)
else:
    print("正式训练尚未运行：", FULL_OUTPUT)


$ rocm-smi --showuse --showmemuse --showtemp


============================ ROCm System Management Interface ============================
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 31.0
=================================== % time GPU is busy ===================================
GPU[0]		: GPU use (%): 0
=================================== Current Memory Use ===================================
GPU[0]		: GPU Memory Allocated (VRAM%): 9
GPU[0]		: Memory Activity: N/A
GPU[0]		: Not supported on the given system
================================== End of ROCm SMI Log ===================================
正式训练尚未运行： $MODEL_ROOT/smolvla_rocm_full


训练完成后不要只挑 loss 最低的节点。下一步进入 `11_mujoco_closed_loop_deploy.ipynb`，用固定 seed、严格 `physical_success` 和视频复核比较中间 checkpoint 与最终 checkpoint。


## 已完成训练的日志快照


In [10]:
progress = json.loads(
    (ASSET_DIR / "training_progress_snapshot.json").read_text(encoding="utf-8")
)
records = [row for row in progress["records"] if row["model"] == 'SmolVLA']
for row in records:
    ratio = row["current_step"] / max(1, row["total_steps"])
    filled = round(36 * ratio)
    bar = "#" * filled + "." * (36 - filled)
    print(f'{row["model"]:>18} [{bar}] {row["current_step"]}/{row["total_steps"]}')
    print("stage:", row["stage"])
    print("closed-loop physical_success:", row["physical_success"])
    print("note:", row["note"])


           SmolVLA [##################..................] 500/1000
stage: weighted blue2 step500
closed-loop physical_success: 53/60
note: step500 的红蓝杯平衡优于 step1000，因此保留中间 checkpoint


![AMD ROCm 历史训练进度与闭环结果](../assets/training_progress_overview.png)

                这张图来自已经完成的 AMD 实验日志，不是假装本次打开 Notebook 后重新跑完的训练。上面的 smoke/full 单元提供真实启动代码；运行长训练时，终端进度会继续写入输出目录，完成后再用 11 的闭环评估决定 checkpoint 是否可用。


## 在 Notebook 中跟踪后台训练日志


In [11]:
def tail_training_log(log_path, lines=25):
    log_path = Path(log_path)
    if not log_path.exists():
        print("训练日志尚不存在：", log_path)
        return
    content = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(content[-lines:]))


TRAIN_LOG = Path(os.environ.get("TRAIN_LOG", FULL_OUTPUT / "train.log"))
print("训练运行后可重复执行：tail_training_log(TRAIN_LOG)")
tail_training_log(TRAIN_LOG)


训练运行后可重复执行：tail_training_log(TRAIN_LOG)
训练日志尚不存在： $MODEL_ROOT/smolvla_rocm_full/train.log
